# Aria Accessibility & WCAG Compliance Guide

**Comprehensive guide for building accessible applications meeting WCAG 2.1 Level AA standards.**

Learn how to make Aria accessible for all users, including those with disabilities.


## WCAG 2.1 Framework

### Accessibility Principles

```
POUR Framework
├─ Perceivable
│  ├─ Provide text alternatives for images
│  ├─ Ensure sufficient color contrast
│  └─ Avoid flashing content
│
├─ Operable
│  ├─ Keyboard navigation support
│  ├─ No keyboard traps
│  └─ Skip to main content
│
├─ Understandable
│  ├─ Clear language (reading level 8-9)
│  ├─ Consistent navigation
│  └─ Error prevention and recovery
│
└─ Robust
   ├─ Valid HTML/ARIA
   ├─ Compatible with assistive technology
   └─ Mobile-friendly
```

### WCAG Conformance Levels

```
Level A (Basic)
├─ Level AA (Mid) - Recommended
│  └─ Level AAA (Enhanced)

Target: WCAG 2.1 Level AA
├─ Text contrast: 4.5:1 (normal), 3:1 (large)
├─ Keyboard accessible
├─ No seizure-inducing content
├─ Meaningful headings and labels
└─ Form error handling
```

---

## Frontend Accessibility

### React Accessibility Patterns

```typescript
// apps/chat/src/components/AccessibleChatInput.tsx
import { useRef, useState } from 'react';

export function AccessibleChatInput() {
  const inputRef = useRef<HTMLInputElement>(null);
  const [error, setError] = useState<string | null>(null);

  const handleSubmit = (e: React.FormEvent) => {
    e.preventDefault();
    const value = inputRef.current?.value;

    if (!value || value.trim().length === 0) {
      setError("Message cannot be empty. Please enter text before sending.");
      // Focus on input to announce error
      inputRef.current?.focus();
      return;
    }

    // Process message
    setError(null);
  };

  return (
    <form onSubmit={handleSubmit} aria-label="Chat message form">
      {/* Error message announced to screen readers */}
      {error && (
        <div
          role="alert"
          aria-live="assertive"
          className="error-message"
        >
          {error}
        </div>
      )}

      <label htmlFor="chat-input">
        Your message
        <span aria-label="required" className="required">*</span>
      </label>

      <textarea
        id="chat-input"
        ref={inputRef}
        placeholder="Type your message and press Enter to send"
        aria-required="true"
        aria-describedby="char-count"
        maxLength={500}
        onKeyDown={(e) => {
          if (e.key === "Enter" && e.ctrlKey) {
            handleSubmit(e as any);
          }
        }}
      />

      <div id="char-count" className="sr-only">
        {inputRef.current?.value.length || 0} of 500 characters
      </div>

      <button
        type="submit"
        aria-label="Send message (Ctrl+Enter)"
      >
        Send
      </button>
    </form>
  );
}
```

### Keyboard Navigation

```typescript
// Implement focus trap in modal
export function AccessibleModal({ isOpen, onClose, children }) {
  const modalRef = useRef<HTMLDivElement>(null);
  const firstFocusableRef = useRef<HTMLButtonElement>(null);
  const lastFocusableRef = useRef<HTMLButtonElement>(null);

  const handleKeyDown = (e: React.KeyboardEvent) => {
    if (e.key === "Escape") {
      onClose();
      return;
    }

    // Focus trap: keep focus within modal
    if (e.key === "Tab") {
      if (e.shiftKey) {
        // Shift+Tab on first element
        if (document.activeElement === firstFocusableRef.current) {
          lastFocusableRef.current?.focus();
          e.preventDefault();
        }
      } else {
        // Tab on last element
        if (document.activeElement === lastFocusableRef.current) {
          firstFocusableRef.current?.focus();
          e.preventDefault();
        }
      }
    }
  };

  if (!isOpen) return null;

  return (
    <div
      role="dialog"
      aria-modal="true"
      ref={modalRef}
      onKeyDown={handleKeyDown}
    >
      <button
        ref={firstFocusableRef}
        onClick={onClose}
        aria-label="Close modal"
      >
        ✕
      </button>

      {children}

      <button
        ref={lastFocusableRef}
        onClick={onClose}
      >
        Done
      </button>
    </div>
  );
}
```

### Color Contrast

```typescript
// Accessibility utility
export class ColorA11y {
    /**
     * Calculate WCAG contrast ratio
     * Formula: (L1 + 0.05) / (L2 + 0.05)
     * where L is relative luminance
     */
    static getContrastRatio(color1: string, color2: string): number {
        const lum1 = this.getRelativeLuminance(color1)
        const lum2 = this.getRelativeLuminance(color2)
        const lighter = Math.max(lum1, lum2)
        const darker = Math.min(lum1, lum2)
        return (lighter + 0.05) / (darker + 0.05)
    }

    static getRelativeLuminance(color: string): number {
        const [r, g, b] = this.hexToRgb(color)

        const rsRGB = r / 255
        const gsRGB = g / 255
        const bsRGB = b / 255

        const calc = (c: number) =>
            c <= 0.03928 ? c / 12.92 : Math.pow((c + 0.055) / 1.055, 2.4)

        return (
            0.2126 * calc(rsRGB) + 0.7152 * calc(gsRGB) + 0.0722 * calc(bsRGB)
        )
    }

    static isWCAGAA(color1: string, color2: string, fontSize: number): boolean {
        const ratio = this.getContrastRatio(color1, color2)
        // AA requires 4.5:1 for normal, 3:1 for large (18pt+)
        return fontSize >= 18 ? ratio >= 3 : ratio >= 4.5
    }

    static isWCAGAAA(
        color1: string,
        color2: string,
        fontSize: number,
    ): boolean {
        const ratio = this.getContrastRatio(color1, color2)
        // AAA requires 7:1 for normal, 4.5:1 for large
        return fontSize >= 18 ? ratio >= 4.5 : ratio >= 7
    }

    private static hexToRgb(hex: string): [number, number, number] {
        const result = /^#?([a-f\d]{2})([a-f\d]{2})([a-f\d]{2})$/i.exec(hex)
        return result
            ? [
                  parseInt(result[1], 16),
                  parseInt(result[2], 16),
                  parseInt(result[3], 16),
              ]
            : [0, 0, 0]
    }
}

// Usage
const isAccessible = ColorA11y.isWCAGAA("#000000", "#FFFFFF", 14)
console.log("AA Compliant:", isAccessible)
```

---

## Backend Accessibility

### API Response Accessibility

```python
# shared/accessibility.py
from dataclasses import dataclass
from typing import Optional

@dataclass
class AccessibleResponse:
    """Response with accessibility annotations."""
    primary_content: str
    screen_reader_summary: str  # For blind users
    keyboard_instructions: str
    skip_links: list  # Links to skip repetitive content
    error_messages: list[dict]  # Clear, constructive errors
    alternatives: dict  # Alt text, descriptions

    def to_dict(self):
        return {
            "content": self.primary_content,
            "aria": {
                "summary": self.screen_reader_summary,
                "keyboardHelp": self.keyboard_instructions,
                "skipLinks": self.skip_links
            },
            "errors": self.error_messages,
            "alt": self.alternatives
        }

# API endpoint with accessibility
@app.route("/api/chat/completions", methods=["POST"])
async def chat_with_accessibility(req: func.HttpRequest):
    """Chat endpoint with accessibility metadata."""
    data = req.get_json()
    message = data.get("message")

    # Validate input
    if not message or not message.strip():
        return func.HttpResponse(
            json.dumps({
                "error": "Message is empty",
                "aria": {
                    "summary": "Error: Empty message. Please enter text before sending.",
                    "role": "alert"
                }
            }),
            status_code=400
        )

    # Generate response
    response_text = await get_chat_response(message)

    # Create accessible response
    accessible_response = AccessibleResponse(
        primary_content=response_text,
        screen_reader_summary=generate_summary(response_text),
        keyboard_instructions="Press Tab to navigate. Enter to interact.",
        skip_links=[
            {"href": "#main-content", "label": "Skip to main content"},
            {"href": "#chat-history", "label": "Jump to chat history"}
        ],
        error_messages=[],
        alternatives={
            "aria-label": "Chat response",
            "aria-live": "polite"
        }
    )

    return func.HttpResponse(
        json.dumps(accessible_response.to_dict()),
        status_code=200
    )
```

---

## Testing for Accessibility

### Automated Testing

```python
# scripts/test_accessibility.py
from axe_selenium_python import Axe

def test_accessibility():
    """Automated accessibility testing with axe."""
    from selenium import webdriver

    driver = webdriver.Chrome()
    driver.get("http://localhost:3000")

    # Run axe accessibility tests
    axe = Axe(driver)
    axe.inject()
    axe.run()

    results = axe.results()

    # Check for violations
    if results["violations"]:
        print("Accessibility violations found:")
        for violation in results["violations"]:
            print(f"  - {violation['description']}")
            print(f"    Impact: {violation['impact']}")
            print(f"    Elements affected: {len(violation['nodes'])}")
    else:
        print("✓ No accessibility violations found")

    driver.quit()
```

### Manual Testing Checklist

```yaml
# tests/accessibility_checklist.yaml
manual_tests:
    keyboard_navigation:
        - "Can navigate all interactive elements using Tab key?"
        - "Can reach all links without tabbing too many times?"
        - "Is focus visible with clear indicator?"
        - "No keyboard traps (can Tab away from all elements)?"

    screen_reader:
        - "Headings properly nested (h1, h2, h3)?"
        - "Links have descriptive text (not 'click here')?"
        - "Images have meaningful alt text?"
        - "Form labels associated with inputs?"
        - "Errors announced to screen readers?"

    visual:
        - "Sufficient color contrast (4.5:1)?"
        - "Text resizable without breaking layout?"
        - "Readable at 200% zoom?"
        - "No reliance on color alone to convey meaning?"

    content:
        - "Plain language (Flesch-Kincaid 8-9)?"
        - "Bulleted lists for grouped items?"
        - "Short paragraphs (3-4 sentences max)?"
        - "Consistent terminology?"

    motion:
        - "Animations can be disabled?"
        - "No flashing content (>3 per second)?"
        - "Parallax effects don't interfere?"
        - "Respects prefers-reduced-motion?"
```

---

## Accessibility Best Practices

### Frontend

- [ ] Semantic HTML (`<button>`, `<nav>`, `<main>`)
- [ ] Proper heading hierarchy (h1, h2, h3)
- [ ] Image alt text (descriptive, not "image of")
- [ ] Form labels and error messages
- [ ] ARIA labels for icon buttons
- [ ] Focus management in modals
- [ ] Keyboard shortcuts documented
- [ ] Color contrast ≥ 4.5:1 (WCAG AA)
- [ ] Responsive text sizing
- [ ] Support for prefers-reduced-motion

### Backend

- [ ] Clear, constructive error messages
- [ ] Structured error responses
- [ ] API documentation in plain language
- [ ] Pagination support (don't require "load more")
- [ ] Rate limiting generous for assistive tech

### Operations

- [ ] Automated accessibility testing in CI
- [ ] Manual accessibility audit quarterly
- [ ] Screen reader testing (NVDA, JAWS)
- [ ] User testing with people with disabilities
- [ ] Accessibility statement on website
- [ ] Feedback form for accessibility issues
